# Slovenia Solvency reports Table S.02.01.02; Part 1 Extraction

The scope of this script is to transcribe SFCR table S.02.01.02 for the 7 life insurance companies on the Slovenian market. The notebook is organized into the following sections:

1) Companies and tables in scope
2) Packages and tools
3) Generation of DataFrames

## Companies in scope

For the year 2024, the companies in scope are the following:

 - Zavarovalnica Triglav d.d.
 - Generali zavarovalnica d.d.
 - Prva osebna zavarovalnica, d.d.
 - Modra zavarovalnica d.d.
 - Vzajemna d.d.
 - Grawe zavarovalnica d.d.
 - Zavarovalnica Sava, d.d.

### Solvency and Financial condition reports
According to Article 51 of the Solvency II Directive 2009/138/EC, companies under the regulatory umbrella of EIOPA, companies must publish annually a Solvency and Financial Condition Reports (SFCR) for all legal entities.

Part of the report is mandatory tables that show some financial and actuarial indicators. One such table is S.02.01.02 which shows a simplified balance sheet of the legal entity. This table is inside the scope of this demo.

## Description of the process

The process of extraction is performed in 5 phases:

### Phase 1: Find the reports and identify the relevant tables. 
 1) Identify the new SFCR report and save it into the folder Input.
 2) Identify the pages where the tables of interest are.
 3) Compile the map of the company run in the master_list.csv.

### Phase 2: Run the Extraction script (this script). 
The script performs the following steps (with slight modifications depending on the table format):
 1) Save the page with the table into a separate folder Single_pdf.
 2) Use either a Python package or specialized LLM to create a digital equivalent of the table.
 3) Fix the systemic errors that prevent the table from being saved as DataFrame.
 4) Save the DataFrame into the Output folder.

### Phase 3: Run the Processing script. 
The script applies fixes to the DataFrame to make the numbers closer to the reported numbers. It joins all the tables into a single dataset. 

### Phase 4: Run the Cross-Validation script. 
Applies a series of tests that check for the internal consistency between the numbers. Flags the potential errors.

### Phase 5: Final modifications to the table and a manual inspection. 

## Necessary Python packages

In [2]:
pip install PyPDF2 pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install mistralai

In [4]:
!pip install pycryptodome

## Python packages

In [5]:
import re
import os
import json
import base64
import pdfplumber
import pandas as pd
import numpy as np
from pathlib import Path
from PyPDF2 import PdfReader, PdfWriter
from mistralai.models import OCRResponse
from IPython.display import Markdown, display
from mistralai import Mistral, DocumentURLChunk, ImageURLChunk, TextChunk

## API key

The model of choice for this project is Mistral. This was identified as the most economical choice. Additionally, Mistral released a custom model for OCR tasks. 

Useful links:

 - https://docs.mistral.ai/getting-started/quickstart/

 - https://mistral.ai/news/mistral-ocr

In [115]:
#api_key = "[YOUR MISTRAL API KEY]"

## Functions

In [7]:
def set_code_index_and_save(table: pd.DataFrame, path: str) -> None:
    """
    Set the 'CODE' column as the index of a DataFrame and save it as a CSV file.
    """
    table = table.set_index("CODE")
    table.to_csv(path)

In [8]:
def extract_tables_from_pdf(pdf_path: str, page_number:int=1) -> pd.DataFrame:
    """
    Extract the first table from a specific page of a PDF and return it as a DataFrame.

    Parameters
    ----------
    pdf_path : str or pathlib.Path
        Path to the PDF file.
    page_number : int, optional (default=1)
        The page number to extract the table from (1-indexed, i.e., first page is 1).

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the extracted table. The first row of the table is
        treated as the header, and the remaining rows as data.

    """
    
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_number - 1]  # Pages are zero-indexed
        tables = page.extract_tables()
        df_tables = pd.DataFrame(tables[0][1:], columns=tables[0][0])
        return df_tables

In [9]:
def extract_page(input_pdf_path: str, output_pdf_path: str, page_number: int, password: str = "")-> None:
    """
    Extract a single page from a PDF file and save it as a new PDF.
    """
    
    pdf_reader = PdfReader(input_pdf_path)
    pdf_writer = PdfWriter()

    # Decrypt if necessary
    if pdf_reader.is_encrypted:
        if password:
            pdf_reader.decrypt(password)
        else:
            pdf_reader.decrypt("")  # try empty password
    
    # Add the specified page to the PdfWriter object
    pdf_writer.add_page(pdf_reader.pages[page_number - 1])

    # Write the selected page to a new PDF file
    with open(output_pdf_path, 'wb') as output_pdf_file:
        pdf_writer.write(output_pdf_file)

In [10]:
def encode_pdf(pdf_path: str)-> None:
    """Encode the pdf to base64."""
    try:
        with open(pdf_path, "rb") as pdf_file:
            return base64.b64encode(pdf_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {pdf_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None

In [11]:
def markdown_table_to_dataframe(markdown_text, shift_row = 0):
    """
    Convert a Markdown-formatted table into a pandas DataFrame.
    """
    
    # Split the markdown text into lines
    lines = markdown_text.strip().split('\n')

    # Filter out lines that are part of the table
    table_lines = [line for line in lines if line.startswith('|')]

    # Remove the markdown table syntax
    cleaned_lines = [line.strip('|').strip() for line in table_lines]

    # Split each line into columns
    data = [line.split('|') for line in cleaned_lines]

    # Extract headers and rows
    headers = [header.strip() for header in data[1+shift_row] if header.strip()]
    rows = [[cell.strip() for cell in row if cell.strip()] for row in data[3:]]

    # Ensure each row has the same number of columns as the headers
    for row in rows:
        if len(row) < len(headers):
            row += [''] * (len(headers) - len(row))

    # Create a DataFrame
    df = pd.DataFrame(rows, columns=headers)
    return df


In [12]:
def split_strings_to_df(strings):
    """
    Convert a list of strings in the format `"text ..... number"` into a DataFrame.

    Each string is expected to contain a text label followed by dots and a numeric value.
    The numeric value is cleaned (spaces and dots removed) and converted to either
    `int` or `float`. If no numeric value is found, `None` is used instead.

    Parameters
    ----------
    strings : list of str
        A list of strings to be parsed.  
        Example: ["Technical provisions ..... 2.337.991", "Best Estimate ..... 0"]

    Returns
    -------
    pd.DataFrame
        A DataFrame with two columns:
        - "NAME": str, the extracted text (trimmed of dots and whitespace).
        - "C0010": int, float, or None, the extracted numeric value.

    Notes
    -----
    - Numbers with thousand separators like `"2.337.991"` are cleaned into integers (`2337991`).
    - If the number cannot be parsed, it remains as a string.
    - Strings without a numeric value will have `None` in the `"C0010"` column.
    """

    data = []
    for s in strings:
        # Try to match "text ..... number"
        match = re.match(r'^(.*?)\.{2,}\s*([\d\s\.]+)$', s.strip())
        if match:
            left = match.group(1).strip()
            # Clean up number (remove spaces in middle, convert to float/int)
            right = match.group(2).replace(" ", "").replace(".", "")
            if right.isdigit():
                right = int(right)
            else:
                try:
                    right = float(right)
                except:
                    pass
            data.append((left, right))
        else:
            # No number found, just keep text
            data.append((s.strip(), None))

    df = pd.DataFrame(data, columns=["NAME", "C0010"])
    return df

In [13]:
def run_mistral_ocr(output_pdf_path: str, api_key: str)-> str:
    """
    Run OCR on a PDF file using the Mistral OCR API and return the extracted text in Markdown format.

    The function:
    1. Encodes the PDF as a base64 string.
    2. Sends it to the Mistral OCR API.
    3. Extracts the recognized text in Markdown format from the response.
    """
    
    # Getting the base64 string
    base64_pdf = encode_pdf(Path(output_pdf_path))
    
    client = Mistral(api_key=api_key)
    
    ocr_response = client.ocr.process(
        model="mistral-ocr-latest",
        document={
            "type": "document_url",
            "document_url": f"data:application/pdf;base64,{base64_pdf}" 
        },
        include_image_base64=True
    )
    
    # Assuming ocr_response is your instance of OCRResponse
    markdown_texts = [page.markdown for page in ocr_response.pages]
    
    # If you want to concatenate all markdown texts from all pages
    full_markdown_text = "\n".join(markdown_texts)
    
    # Assuming ocr_response is your OCRResponse object
    markdown_text = ocr_response.pages[0].markdown
    return markdown_text
    

In [14]:
def parse_table_to_df(text: str, value_column_name:str) -> pd.DataFrame:
    """
    Parse a long text table into a DataFrame with columns:
    ['code', 'name', 'value'].
    """
    # Split text into lines
    lines = text.splitlines()
    
    rows = []
    buffer = []
    for line in lines:
        # Match lines with pattern: code (Rxxxx) followed by number
        match = re.match(r"^(R\d{4})\s*(.*)$", line)
        if match:
            code = match.group(1)
            rest = match.group(2).strip()
            
            # Check if 'rest' is a number (the value)
            if re.match(r"^[\d\.,]+$", rest):
                name = " ".join(buffer).strip()
                value = rest
                rows.append((code, name, value))
                buffer = []  # reset for next name
            else:
                # It's not just a number, so treat it as name continuation
                buffer.append(line)
        else:
            # If line is just a number after a code line
            if re.match(r"^[\d\.,]+$", line.strip()):
                name = " ".join(buffer).strip()
                value = line.strip()
                # Last buffer entry should contain the code
                code_match = re.search(r"(R\d{4})", buffer[-1]) if buffer else None
                code = code_match.group(1) if code_match else None
                # Remove code from name
                buffer[-1] = buffer[-1].replace(code, "").strip() if code else buffer[-1]
                rows.append((code, " ".join(buffer).strip(), value))
                buffer = []  # reset
            else:
                # Accumulate description
                buffer.append(line)
    
    # Build DataFrame
    df = pd.DataFrame(rows, columns=["CODE", "NAME", value_column_name])
    
    # Convert numeric values to float
    df[value_column_name] = df[value_column_name].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    df[value_column_name] = pd.to_numeric(df[value_column_name], errors="coerce")
    
    return df

In [15]:
def extract_paths(master_list: pd.DataFrame, unique_id: str):
    """
    Extract file paths and page number from master_list for a given unique_id.
    
    Parameters:
        master_list (pd.DataFrame): DataFrame containing metadata.
        unique_id: Index or identifier in the DataFrame.
    
    Returns:
        dict: Dictionary with keys 'pdf_path', 'page_number', 
              'output_pdf_path', 'output_final_path', 'codes_path'.
    """
    return master_list.loc[unique_id, "document_name"], int(master_list.loc[unique_id, "page_number"]), master_list.loc[unique_id, "output_pdf_path"], master_list.loc[unique_id, "output_final_path"], master_list.loc[unique_id, "codes_path"]

In [16]:
def convert_to_dataframe(text: str) -> pd.DataFrame:
    """
    Convert raw AnnexI/Solvency II balance sheet text into a structured DataFrame.

    Parameters
    ----------
    text : str
        Raw text containing balance sheet items.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: CODE, NAME, VALUE
    """
    rows = []
    
    # Split text into lines
    for line in text.splitlines():
        # Match rows with a code like R001 and a value at the end
        match = re.match(r"^(.*)\s+(R\d+)\s+([-]?\d[\d\.\,]*)$", line.strip())
        if match:
            name = match.group(1).strip()
            code = match.group(2).strip()
            value_str = match.group(3).replace(".", "")  # remove thousand separators
            value = float(value_str.replace(",", "."))   # convert to float
            rows.append((code, name, value))
    
    df = pd.DataFrame(rows, columns=["CODE", "NAME", "VALUE"])
    return df

In [17]:
def append_zero_if_len4(df: pd.DataFrame, col: str = "CODE") -> pd.DataFrame:
    """
    Append '0' to the code if its length is 4 characters.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing the code column.
    col : str
        Column name for codes (default = 'CODE').

    Returns
    -------
    pd.DataFrame
        DataFrame with modified codes.
    """
    df = df.copy()
    df[col] = df[col].apply(lambda x: x + "0" if len(x) == 4 else x)
    return df

In [18]:
def run_ocr_and_convert_to_df(path: str, api_key: str)-> pd.DataFrame:
    """
    Run OCR on a PDF file and convert the extracted Markdown table into a DataFrame.
    """
    
    markdown_text = run_mistral_ocr(path, api_key)
    table = markdown_table_to_dataframe(markdown_text)
    return table

In [19]:
def ocr_to_dataframe(ocr_text: str) -> pd.DataFrame:
    """
    Converts OCR extracted financial text into a structured DataFrame.
    
    Extracts description, code (Rxxxx, Cxxxx), and numeric value if present.
    """
    rows = []
    
    # Split text into lines
    for line in ocr_text.splitlines():
        # Look for patterns like "..... R0030 ..... 123.456"
        match = re.match(r"^(.*?)\.{2,}\s*(R\d{4}|C\d{4})(?:\s*\.{2,}\s*([\d\.\s-]+))?$", line.strip())
        if match:
            desc = match.group(1).strip()
            code = match.group(2).strip()
            raw_value = match.group(3)
            
            if raw_value:
                # Remove spaces inside numbers (e.g. "4.275 .590" -> "4.275.590")
                cleaned = raw_value.replace(" ", "")
                # Convert to float if numeric, else keep as string
                try:
                    value = float(cleaned.replace(".", "").replace(",", "."))
                except ValueError:
                    value = cleaned
            else:
                value = None
            
            rows.append((code, desc, value))
    
    df = pd.DataFrame(rows, columns=["CODE", "DESCRIPTION", "VALUE"])
    return df

In [20]:
def ocr_text_to_dataframe(text: str) -> pd.DataFrame:
    """
    Transforms OCR-extracted text containing '.....' separators into a clean DataFrame.

    Parameters
    ----------
    text : str
        OCR-extracted text containing lines with labels and numeric values.

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns ['Description', 'Value'].
        - Description: textual part
        - Value: numeric value (float) or NaN if missing
    """

    # Step 1: Clean up spaces and join wrapped lines
    text = re.sub(r'\s*\n\s*', '\n', text.strip())  # normalize line breaks
    text = re.sub(r' +', ' ', text)  # collapse multiple spaces
    text = text.replace('\u202f', '').replace('\xa0', '')  # remove non-breaking spaces

    # Step 2: Split text into lines
    lines = text.split("\n")

    # Step 3: Merge broken lines (those without numeric values)
    merged_lines = []
    buffer = ""
    for line in lines:
        # if line has no number, append it to previous description
        if not re.search(r'\d', line):
            buffer += " " + line.strip()
        else:
            if buffer:
                line = buffer.strip() + " " + line.strip()
                buffer = ""
            merged_lines.append(line.strip())

    # Step 4: Extract description and numeric value
    data = []
    for line in merged_lines:
        # Match pattern like "text ..... number"
        match = re.match(r"^(.*?)\.{2,}\s*([\d\.\s]+)$", line)
        if match:
            desc, val = match.groups()
        else:
            # Try to separate trailing numbers without dots
            match = re.match(r"^(.*?)([\d\.\s]+)$", line)
            if match:
                desc, val = match.groups()
            else:
                desc, val = line, np.nan

        # Clean description and numeric value
        desc = desc.strip(" .")
        if isinstance(val, str):
            val = re.sub(r"[^\d,\.]", "", val)
            val = val.replace(" ", "").replace(",", ".")
            try:
                val = float(val)
            except ValueError:
                val = np.nan

        data.append((desc, val))

    df = pd.DataFrame(data, columns=["Description", "Value"])
    return df

In [21]:
def parse_table_to_df(text: str, value_column_name:str) -> pd.DataFrame:
    """
    Parse a long text table into a DataFrame with columns:
    ['code', 'name', 'value'].
    """
    # Split text into lines
    lines = text.splitlines()
    
    rows = []
    buffer = []
    for line in lines:
        # Match lines with pattern: code (Rxxxx) followed by number
        match = re.match(r"^(R\d{4})\s*(.*)$", line)
        if match:
            code = match.group(1)
            rest = match.group(2).strip()
            
            # Check if 'rest' is a number (the value)
            if re.match(r"^[\d\.,]+$", rest):
                name = " ".join(buffer).strip()
                value = rest
                rows.append((code, name, value))
                buffer = []  # reset for next name
            else:
                # It's not just a number, so treat it as name continuation
                buffer.append(line)
        else:
            # If line is just a number after a code line
            if re.match(r"^[\d\.,]+$", line.strip()):
                name = " ".join(buffer).strip()
                value = line.strip()
                # Last buffer entry should contain the code
                code_match = re.search(r"(R\d{4})", buffer[-1]) if buffer else None
                code = code_match.group(1) if code_match else None
                # Remove code from name
                buffer[-1] = buffer[-1].replace(code, "").strip() if code else buffer[-1]
                rows.append((code, " ".join(buffer).strip(), value))
                buffer = []  # reset
            else:
                # Accumulate description
                buffer.append(line)
    
    # Build DataFrame
    df = pd.DataFrame(rows, columns=["CODE", "DESCRIPTION", value_column_name])
    
    # Convert numeric values to float
    df[value_column_name] = df[value_column_name].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    df[value_column_name] = pd.to_numeric(df[value_column_name], errors="coerce")
    
    return df

## The list of companies



In [22]:
master_list = pd.read_csv("master_list.csv", header=0, index_col=0)

In [23]:
display(master_list)

,company,document_name,table_name,page_number,output_pdf_path,output_final_path,codes_path,leto
VZAJEMNA,,,,,,,,
GENERALI_02_1,GENERALI,Input\Porocilo-o-solventnosti-in-financnem-pol...,S.02.01.02_A,83,Single_pdf/GENERALI_S02_01_02_1_2024.pdf,Output/GENERALI_S02_01_02_1_2024.csv,Codes/Codes_S02_A.csv,2024
GENERALI_02_2,GENERALI,Input\Porocilo-o-solventnosti-in-financnem-pol...,S.02.01.02_L,84,Single_pdf/GENERALI_S02_01_02_2_2024.pdf,Output/GENERALI_S02_01_02_2_2024.csv,Codes/Codes_S02_L.csv,2024
TRIGLAV_02_1,TRIGLAV,Input\Poročilo+o+solventnosti+in+finančnem+pol...,S.02.01.02_A,110,Single_pdf/TRIGLAV_S02_01_02_1_2024.pdf,Output/TRIGLAV_S02_01_02_1_2024.csv,Codes/Codes_S02_A_Triglav.csv,2024
TRIGLAV_02_2,TRIGLAV,Input\Poročilo+o+solventnosti+in+finančnem+pol...,S.02.01.02_L,111,Single_pdf/TRIGLAV_S02_01_02_2_2024.pdf,Output/TRIGLAV_S02_01_02_2_2024.csv,Codes/Codes_S02_L_Triglav.csv,2024
PRVA_02_1,PRVA,Input\Prva-osebna-zavarovalnica-PSFP_2024-2025...,S.02.01.02_A,63,Single_pdf/PRVA_S02_01_02_1_2024.pdf,Output/PRVA_S02_01_02_1_2024.csv,NaN,2024
PRVA_02_2,PRVA,Input\Prva-osebna-zavarovalnica-PSFP_2024-2025...,S.02.01.02_L,64,Single_pdf/PRVA_S02_01_02_2_2024.pdf,Output/PRVA_S02_01_02_2_2024.csv,NaN,2024
MODRA_02_1,MODRA,Input\Revidirano-SFCR-porocilo-2024.pdf,S.02.01.02_A,83,Single_pdf/MODRA_S02_01_02_1_2024.pdf,Output/MODRA_S02_01_02_1_2024.csv,NaN,2024
MODRA_02_2,MODRA,Input\Revidirano-SFCR-porocilo-2024.pdf,S.02.01.02_L,84,Single_pdf/MODRA_S02_01_02_2_2024.pdf,Output/MODRA_S02_01_02_2_2024.csv,NaN,2024
MODRA_02_3,MODRA,Input\Revidirano-SFCR-porocilo-2024.pdf,S.02.01.02_L2,85,Single_pdf/MODRA_S02_01_02_3_2024.pdf,Output/MODRA_S02_01_02_3_2024.csv,NaN,2024


# Code

## GENERALI

#### S.02.01.02 1

In [24]:
unique_id = "GENERALI_02_1"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [25]:
codes_S020102_A = pd.read_csv(codes_path,header = 0, index_col=0)

In [26]:
table = extract_tables_from_pdf(pdf_path=output_pdf_path, page_number=1)

In [27]:
table_e = pd.concat([codes_S020102_A,table], ignore_index=True, axis = 1)

In [28]:
table_e = table_e.iloc[:, [0,1,4]]

In [29]:
table_e.columns = ["DESCRIPTION","CODE", "C0010"]

In [30]:
set_code_index_and_save(table=table_e, path=output_final_path)

In [31]:
del table, table_e

#### S.02.01.02 2

In [32]:
unique_id = "GENERALI_02_2"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [33]:
codes_S020102_L = pd.read_csv(codes_path,header = 0, index_col=0)

In [34]:
table = extract_tables_from_pdf(pdf_path=output_pdf_path, page_number=1)

In [35]:
table = table.drop(index=(38))

In [36]:
table=table.reset_index(drop=True)

In [37]:
table_e = pd.concat([codes_S020102_L,table], ignore_index=True, axis = 1)

In [38]:
table_e = table_e.iloc[:,[0,1,4]]

In [39]:
table_e.columns = ["DESCRIPTION","CODE", "C0010"]

In [40]:
set_code_index_and_save(table=table_e, path=output_final_path)

In [41]:
del table, table_e

## Triglav

#### S.02.01.02 1

In [42]:
unique_id = "TRIGLAV_02_1"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [43]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [44]:
codes_S020102_A = pd.read_csv(codes_path,header = 0, index_col=0)

In [45]:
table = table.drop(index=(25))
table=table.reset_index(drop=True)

In [46]:
table_e = pd.concat([codes_S020102_A,table], ignore_index=True, axis = 1)

In [47]:
table_e = table_e.iloc[:,[0,1,3]]

In [48]:
table_e.columns = ["DESCRIPTION","CODE", "C0010"]

In [49]:
set_code_index_and_save(table=table_e, path=output_final_path)

In [50]:
del table, table_e

#### S.02.01.02 2

In [51]:
unique_id = "TRIGLAV_02_2"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [52]:
markdown_text = run_mistral_ocr(output_pdf_path, api_key)

In [53]:
table = ocr_text_to_dataframe(markdown_text)

In [54]:
codes_S020102_L = pd.read_csv(codes_path,header = 0, index_col=0)

In [55]:
table_e = pd.concat([codes_S020102_L,table], ignore_index=True, axis = 1)

In [56]:
table_e = table_e.iloc[:,[0,1,3]]

In [57]:
table_e.columns = ["DESCRIPTION","CODE", "C0010"]

In [58]:
set_code_index_and_save(table=table_e, path=output_final_path)

## Prva osebna zavarovalnica, d.d.

#### S.02.01.02 1

In [59]:
unique_id = "PRVA_02_1"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [60]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [61]:
table.columns = ["CODE", "DESCRIPTION", "C0010"]

In [62]:
set_code_index_and_save(table=table, path=output_final_path)

In [63]:
del table

#### S.02.01.02 2

In [64]:
unique_id = "PRVA_02_2"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [65]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [66]:
table.columns = ["CODE", "DESCRIPTION", "C0010"]

In [67]:
set_code_index_and_save(table=table, path=output_final_path)

In [68]:
del table

## Modra zavarovalnica d.d.

#### S.02.01.02 1

In [69]:
unique_id = "MODRA_02_1"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [70]:
markdown_text = run_mistral_ocr(output_pdf_path, api_key)

In [71]:
table = parse_table_to_df(markdown_text, "C0010")

In [72]:
set_code_index_and_save(table=table, path=output_final_path)

In [73]:
del table

#### S.02.01.02 2

In [74]:
unique_id = "MODRA_02_2"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [75]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [76]:
table.columns = ["DESCRIPTION", "CODE", "C0010"]

In [77]:
set_code_index_and_save(table=table, path=output_final_path)

In [78]:
del table

#### S.02.01.02 3

In [79]:
unique_id = "MODRA_02_3"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [80]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [81]:
table.columns = ["DESCRIPTION", "CODE", "C0010"]

In [82]:
set_code_index_and_save(table=table, path=output_final_path)

In [83]:
del table

## Vzajemna d.d.

#### S.02.01.02 1

In [84]:
unique_id = "VZAJEMNA_02_1"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [85]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [86]:
table.columns = ["DESCRIPTION", "CODE", "C0010"]

In [87]:
set_code_index_and_save(table=table, path=output_final_path)

In [88]:
del table

#### S.02.01.02 2

In [89]:
unique_id = "VZAJEMNA_02_2"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [90]:
table = extract_tables_from_pdf(pdf_path=output_pdf_path, page_number=1)

In [91]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [92]:
table.columns = ["DESCRIPTION", "CODE", "C0010"]

In [93]:
set_code_index_and_save(table=table, path=output_final_path)

In [94]:
del table

## Grawe zavarovalnica d.d.

#### S.02.01.02 1

In [95]:
unique_id = "GRAWE_02_1"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [96]:
table = extract_tables_from_pdf(output_pdf_path)

In [97]:
table.columns = ["DESCRIPTION","CODE", "C0010"]

In [98]:
set_code_index_and_save(table=table, path=output_final_path)

In [99]:
del table

#### S.02.01.02 2

In [100]:
unique_id = "GRAWE_02_2"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [101]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [102]:
table.columns = ["DESCRIPTION","CODE", "C0010"]

In [103]:
set_code_index_and_save(table=table, path=output_final_path)

In [104]:
del table

## Zavarovalnica Sava, d.d.

#### S.02.01.02 1

In [105]:
unique_id = "SAVA_02_1"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [106]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [107]:
table.columns = ["CODE","DESCRIPTION", "C0010"]

In [108]:
set_code_index_and_save(table=table, path=output_final_path)

In [109]:
del table

#### S.02.01.02 2

In [110]:
unique_id = "SAVA_02_2"
pdf_path, page_number, output_pdf_path, output_final_path, codes_path =  extract_paths(master_list=master_list, unique_id=unique_id)
extract_page(input_pdf_path=pdf_path, output_pdf_path=output_pdf_path, page_number=page_number, password = "")

In [111]:
table = run_ocr_and_convert_to_df(path=output_pdf_path, api_key=api_key)

In [112]:
table.columns = ["CODE", "DESCRIPTION", "C0010"]

In [113]:
set_code_index_and_save(table=table, path=output_final_path)

In [114]:
del table